# Data Acquisition (XJTU Downsampling)
**Goal:** Aggregate and downsample the raw XJTU vibration data so that 1 CSV file = 1 Bearing (where 1 row represents 1 minute of operation).

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from tqdm import tqdm

# ==========================================
# CONFIGURATION
# ==========================================
# Define input and output paths (Modify as necessary)
RAW_DATA_PATH = r"../Raw_Data/XJTU-SY" 
OUTPUT_PATH = r"../Processed_Data/Downsampled"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Dataset Specifics
# XJTU: 25.6 kHz, ~1.28 sec snapshot (32,768 rows) every 1 minute.
# We will simply load and structure these files, potentially extracting 
# baseline representations (like saving it for feature extraction later).
ROWS_PER_FILE = 32768

print(f"Raw Data Path: {os.path.abspath(RAW_DATA_PATH)}")
print(f"Output Path: {os.path.abspath(OUTPUT_PATH)}")

In [ ]:
def process_and_combine_bearing_data():
    """
    Recursively finds directories containing CSV files in RAW_DATA_PATH.
    Each folder containing CSVs is treated as a bearing dataset.
    Aggregates the 1-minute CSV arrays into 1 CSV per bearing.
    """
    if not os.path.exists(RAW_DATA_PATH):
        print(f"Warning: Raw data path '{RAW_DATA_PATH}' does not exist. Please double-check.")
        return

    # Collect all directories that actually contain .csv files
    bearing_dirs = []
    for root, dirs, files in os.walk(RAW_DATA_PATH):
        if any(f.endswith('.csv') for f in files):
            bearing_dirs.append(root)
            
    if not bearing_dirs:
        print(f"Error: No CSV files found inside any subdirectory of {RAW_DATA_PATH}")
        return
        
    print(f"Found {len(bearing_dirs)} Bearing directories to process.")
    
    for b_dir in tqdm(bearing_dirs, desc="Total Bearings Processed"):
        bearing_id = os.path.basename(b_dir) # e.g., Bearing1_1
        csv_files = sorted(glob.glob(os.path.join(b_dir, "*.csv")))
        
        # Prepare a list to collect data
        bearing_data_list = []
        
        # Extract minute information from sorted CSVs
        for minute_idx, file in enumerate(tqdm(csv_files, desc=f"Reading {bearing_id}", leave=False)):
            try:
                # Load the raw minute file
                df_min = pd.read_csv(file)
                
                # Extract signals
                if 'Horizontal_vibration_signals' in df_min.columns:
                    h_sig = df_min['Horizontal_vibration_signals'].values
                    v_sig = df_min['Vertical_vibration_signals'].values
                else:
                    # Generic fallback if columns differ
                    h_sig = df_min.iloc[:, 0].values
                    v_sig = df_min.iloc[:, 1].values if df_min.shape[1] > 1 else np.zeros_like(h_sig)
                
                # Package 1 row per minute
                minute_record = {
                    'Minute': minute_idx + 1,
                    'H_Signal': ",".join(map(str, h_sig)), # Store as string array for safety
                    'V_Signal': ",".join(map(str, v_sig))
                }
                bearing_data_list.append(minute_record)
            except Exception as e:
                print(f"Error reading file {file}: {e}")
        
        # Convert to DataFrame and Save
        if bearing_data_list:
            final_df = pd.DataFrame(bearing_data_list)
            output_filename = os.path.join(OUTPUT_PATH, f"{bearing_id}_downsampled.csv")
            final_df.to_csv(output_filename, index=False)
            tqdm.write(f"Saved: {output_filename} (Rows/Minutes: {len(final_df)})")

# Execute the pipeline
process_and_combine_bearing_data()